# CSE 455 — Facial Expression Recognition (Experiment A: Baseline CNN)

**Course:** CSE 455 Computer Vision, Autumn 2026
**Team:** Hsu Wai Hnin Kyaw, Larrianna Warner, Xuan Nhu Tran, Rina Yuan
**Project:** Facial Expression Recognition Using Deep Learning: Improving Robustness Under Real-World Conditions

## Goal
Build a deep-learning system that classifies a cropped face image into one of seven emotions — **angry, disgust, fear, happy, sad, surprise, neutral** — and returns per-class confidence scores. Then measure how real-world conditions (lighting, rotation, occlusion) degrade it.

## The 2x2 experiment design
| Experiment | Model | Augmentation |
|---|---|---|
| **A (this notebook)** | CNN trained from scratch | No |
| B | CNN trained from scratch | Yes |
| C | Pretrained ResNet18 | No |
| D | Pretrained ResNet18 | Yes |

**This notebook is Experiment A only:** a baseline CNN trained from scratch, *without* data augmentation, on FER2013. Experiments B, C, and D are separate notebooks that reuse this pipeline — see the last cell for where each one plugs in.

## How to run
1. Runtime → Change runtime type → **GPU** (T4).
2. Run cells top to bottom. The only thing you must supply is Kaggle credentials (section 1) — everything else runs as-is.
3. Checkpoints and results are saved under `./checkpoints` (or your Drive folder if you mount it).

## 0. Setup

In [ ]:
# Install dependencies (torch/torchvision are preinstalled on Colab, but this makes the notebook self-contained).
!pip install -q torch torchvision scikit-learn matplotlib seaborn pandas

In [ ]:
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected — switch to a GPU runtime (Runtime → Change runtime type → T4).')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 1. Get the data — FER2013 from Kaggle

We use the folder version of FER2013 ([msambare/fer2013](https://www.kaggle.com/datasets/msambare/fer2013)): ~35,887 grayscale 48x48 face images, split into `train/` and `test/` folders, each with 7 emotion subfolders. **The official test set stays untouched until final evaluation** — validation comes from a split of the train set (section 3).

**Kaggle credentials — do one of these before running the next cell:**
- **Option A (recommended):** in Colab, click the key icon in the left sidebar (Secrets) and add two secrets: `KAGGLE_USERNAME` (your Kaggle username) and `KAGGLE_KEY` (your API key).
- **Option B:** go to Kaggle → your profile → Settings → **Create New API Token**. That downloads `kaggle.json`. Upload it with the Files panel in Colab, then run `!cp kaggle.json ~/.kaggle/kaggle.json`.

Never commit `kaggle.json` — it is in the repo's `.gitignore`.

In [ ]:
import json
import os

# --- Kaggle credentials (see the markdown cell above) ---
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

try:
    from google.colab import userdata
    creds = {'username': userdata.get('KAGGLE_USERNAME'), 'key': userdata.get('KAGGLE_KEY')}
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump(creds, f)
    print('Kaggle credentials written from Colab Secrets.')
except Exception as e:
    print('Colab Secrets not found — upload kaggle.json and copy it to ~/.kaggle/ :', e)

if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    raise RuntimeError('kaggle.json not found. Add KAGGLE_USERNAME / KAGGLE_KEY as Colab Secrets, '
                       'or upload kaggle.json and run: !cp kaggle.json ~/.kaggle/kaggle.json')
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# --- Download FER2013 (msambare/fer2013) ---
# The zip contains train/ and test/ folders, each with 7 emotion subfolders.
!mkdir -p /content/data
!kaggle datasets download -d msambare/fer2013 -p /content/data --unzip
!ls /content/data
print('Download finished.')

In [ ]:
from collections import Counter
from pathlib import Path

DATA_ROOT = '/content/data'  # change this if your FER2013 folders landed elsewhere

train_dir = Path(DATA_ROOT) / 'train'
test_dir = Path(DATA_ROOT) / 'test'
assert train_dir.is_dir(), f'train/ not found under {DATA_ROOT} — check DATA_ROOT'
assert test_dir.is_dir(), f'test/ not found under {DATA_ROOT} — check DATA_ROOT'

# Class names in alphabetical order — this is also the label index order
# torchvision.datasets.ImageFolder uses (0=angry, 1=disgust, 2=fear, 3=happy,
# 4=neutral, 5=sad, 6=surprise).
CLASS_NAMES = sorted(d.name for d in train_dir.iterdir() if d.is_dir())
print('Classes:', CLASS_NAMES)

for split, d in [('train', train_dir), ('test', test_dir)]:
    counts = {cls: len([p for p in (d / cls).iterdir() if p.is_file()]) for cls in CLASS_NAMES}
    total = sum(counts.values())
    print(f'\n{split}: {total} images')
    for cls in CLASS_NAMES:
        print(f'  {cls:9s} {counts[cls]:5d}  ({counts[cls] / total:.1%})')

## 2. Explore the data

In [ ]:
import random

import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)

fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for j, cls in enumerate(CLASS_NAMES):
    imgs = list((train_dir / cls).iterdir())
    chosen = random.sample(imgs, 2)
    for i, path in enumerate(chosen):
        axes[i, j].imshow(Image.open(path), cmap='gray')
        axes[i, j].set_title(cls, fontsize=10)
        axes[i, j].axis('off')
plt.suptitle('FER2013 samples — 2 per emotion', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

train_counts = {cls: len([p for p in (train_dir / cls).iterdir() if p.is_file()]) for cls in CLASS_NAMES}
test_counts = {cls: len([p for p in (test_dir / cls).iterdir() if p.is_file()]) for cls in CLASS_NAMES}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, counts, title in [(axes[0], train_counts, 'Train'), (axes[1], test_counts, 'Test')]:
    ax.bar(counts.keys(), counts.values())
    ax.set_title(title)
    ax.set_ylabel('Images')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Class distribution — FER2013', fontsize=13)
plt.tight_layout()
plt.show()

print('Train counts:', train_counts)

### What to notice: class imbalance
FER2013 is imbalanced — **disgust is by far the rarest class** (a few hundred images) while **happy is the most common** (several thousand). Expect per-class scores to reflect that: disgust will likely have the lowest recall/F1 even in a good model. That is why the evaluation in section 6 reports per-class precision/recall/F1 and a confusion matrix, not just overall accuracy.

## 3. Preprocessing and train/validation split

- **Validation:** a stratified 90/10 split of the *official train set* (seed 42), so each class keeps its share in both parts. The official test set is never touched in this notebook.
- **Baseline transforms (no augmentation):** `Resize(48)` → `ToTensor()` → `Normalize(mean=0.5, std=0.5)`. FER2013 images are already 48x48 grayscale; `Resize(48)` keeps the pipeline explicit. Normalizing to [-1, 1] helps training converge.
- The validation set gets the *same* transforms here (there is no augmentation to withhold) — but it is built as a separate dataset so that Experiment B can give the train set augmentation transforms while validation stays clean.

In [ ]:
import random

import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Baseline transforms — NO augmentation in Experiment A.
train_transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),                       # -> [0, 1], shape (1, 48, 48)
    transforms.Normalize(mean=[0.5], std=[0.5])  # -> [-1, 1]
])
eval_transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds = datasets.ImageFolder(train_dir, transform=eval_transform)  # same images, eval transform
test_ds = datasets.ImageFolder(test_dir, transform=eval_transform)
print('Classes (index order):', train_ds.classes)

# Stratified 90/10 split of the OFFICIAL train set (seed 42). Official test set stays untouched.
targets = train_ds.targets
train_idx, val_idx = train_test_split(
    range(len(train_ds)), test_size=0.10, random_state=42, stratify=targets
)
train_subset = Subset(train_ds, train_idx)
val_subset = Subset(val_ds, val_idx)
print(f'Train: {len(train_subset)} | Val: {len(val_subset)} | Test (held out): {len(test_ds)}')

BATCH_SIZE = 64
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=torch.cuda.is_available())

images, labels = next(iter(train_loader))
print('Batch:', images.shape, labels.shape)  # expect torch.Size([64, 1, 48, 48]) torch.Size([64])

## 4. Model — baseline CNN

A 3-convolutional-block CNN for 48x48 grayscale input, 7 outputs. Each block is Conv → BatchNorm → ReLU → MaxPool, halving the spatial size:

- Block 1: 48x48 → 24x24 (32 channels)
- Block 2: 24x24 → 12x12 (64 channels)
- Block 3: 12x12 → 6x6 (128 channels)

Flatten gives `128 * 6 * 6 = 4608` features, then two fully-connected layers (4608 → 256 → 7) with dropout 0.5 before the output layer.

In [ ]:
import torch.nn as nn


class BaselineCNN(nn.Module):
    """3-block CNN for 48x48 grayscale faces -> 7 emotions.

    48x48 --pool--> 24x24 --pool--> 12x12 --pool--> 6x6,
    then flatten 128*6*6 = 4608.
    """

    def __init__(self, num_classes=7, dropout_p=0.5):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 48x48 -> 24x24
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Block 2: 24x24 -> 12x12
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Block 3: 12x12 -> 6x6
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = BaselineCNN(num_classes=len(CLASS_NAMES)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {n_params:,}')

# Shape check: a batch of 2 grayscale 48x48 images must come out as (2, 7).
with torch.no_grad():
    out = model(torch.randn(2, 1, 48, 48).to(device))
print('Output shape:', tuple(out.shape))
assert tuple(out.shape) == (2, len(CLASS_NAMES))

## 5. Training

Adam (lr 0.001), CrossEntropyLoss, 15 epochs, batch size 64. The best checkpoint by **validation accuracy** is saved after every epoch that improves it. On a Colab T4 GPU, expect on the order of 10–20 minutes; do not run this on CPU if you can avoid it.

In [ ]:
# --- Checkpoints ---
# Default: ./checkpoints (lost when the Colab runtime is recycled).
# To keep checkpoints on Google Drive instead, uncomment the two lines below
# and set CHECKPOINT_DIR to the Drive path.
# from google.colab import drive
# drive.mount('/content/drive')
CHECKPOINT_DIR = './checkpoints'
# CHECKPOINT_DIR = '/content/drive/MyDrive/emotion-detection/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', CHECKPOINT_DIR)

In [ ]:
import json
import time

import torch.nn as nn
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
NUM_EPOCHS = 15
best_path = os.path.join(CHECKPOINT_DIR, 'best_baseline_cnn.pth')


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc='train', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate_loss_acc(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc='val', leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        running_loss += criterion(outputs, labels).item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

print(f'Training for {NUM_EPOCHS} epochs (Adam lr=0.001, batch={BATCH_SIZE})...')
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate_loss_acc(model, val_loader, criterion, device)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_acc': best_val_acc}, best_path)
    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | train loss {train_loss:.4f} acc {train_acc:.4f} | '
          f'val loss {val_loss:.4f} acc {val_acc:.4f} | {time.time() - t0:.0f}s')

with open(os.path.join(CHECKPOINT_DIR, 'history_baseline.json'), 'w') as f:
    json.dump(history, f, indent=2)
print(f'\nBest val accuracy: {best_val_acc:.4f} — checkpoint saved to {best_path}')

## 6. Evaluation

Load the best checkpoint and evaluate on the validation set. Report per-class precision/recall/F1 (overall accuracy alone hides the imbalance) and plot the normalized confusion matrix plus the training curves.

In [ ]:
from sklearn.metrics import classification_report

best = torch.load(best_path, map_location=device, weights_only=True)
model.load_state_dict(best['model_state_dict'])
print(f"Loaded best checkpoint (epoch {best['epoch']}, val acc {best['val_acc']:.4f})")

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='eval'):
        outputs = model(images.to(device))
        all_preds.extend(outputs.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds, normalize='true')
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion matrix — validation set (rows normalized)')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history['train_loss'], label='train')
axes[0].plot(epochs, history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].plot(epochs, history['train_acc'], label='train')
axes[1].plot(epochs, history['val_acc'], label='val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

## What to record for the report

- **Best validation accuracy** (from the checkpoint) and the epoch it was reached at.
- **Per-class precision / recall / F1** from the classification report — expect disgust to be the weakest (fewest training images).
- **The confusion matrix figure:** which emotions get confused with each other (in FER2013, fear vs surprise and sad vs neutral are common confusions).
- **The training curves:** whether the model overfits (train accuracy keeps rising while validation accuracy plateaus).
- Save the figures: the confusion matrix and curves above, plus `history_baseline.json` in `./checkpoints`.

## Where Experiments B, C, D plug in

- **B (CNN + augmentation):** copy this notebook; in the *train* transform only, add `transforms.RandomHorizontalFlip()`, `transforms.RandomRotation(15)`, and a brightness jitter such as `transforms.ColorJitter(brightness=0.4)`. Validation/test transforms stay exactly as they are.
- **C (ResNet18, no augmentation):** replace `BaselineCNN` with `torchvision.models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)`, set `model.fc = nn.Linear(model.fc.in_features, 7)`, and change the input transform to `Resize((224, 224))` + `Grayscale(num_output_channels=3)` + ImageNet normalization (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`). Everything else (split, training loop, evaluation) stays the same.
- **D (ResNet18 + augmentation):** C with the augmentation transforms from B.
- **Robustness tests (Nov 23 milestone):** evaluate every trained model on distorted copies of the *test* set — dark (brightness ×0.5), bright (×1.5), rotation ±15°/±30°, eye occlusion, mouth occlusion — using the same transforms for all four models.